### Catboost With Additional Data
The same example from my notebook here but the performance is enhanced using additional data from the original dataset.

https://www.kaggle.com/code/erenakbulut/catboost-score-0-87277

#### Dataset Description From This Competition
The dataset for this competition (both train and test) was generated from a deep learning model trained on the Stroke Prediction Dataset. Feature distributions are close to, but not exactly the same, as the original. Feel free to use the original dataset as part of this competition, both to explore differences as well as to see whether incorporating the original in training improves model performance.

#### Dataset Description From The Original Source
Context According to the World Health Organization (WHO) stroke is the 2nd leading cause of death globally, responsible for approximately 11% of total deaths. This dataset is used to predict whether a patient is likely to get stroke based on the input parameters like gender, age, various diseases, and smoking status. Each row in the data provides relavant information about the patient.

Attribute Information

- id: unique identifier
- gender: "Male", "Female" or "Other"
- age: age of the patient
- hypertension: 0 if the patient doesn't have hypertension, 1 if the patient has hypertension
- heart_disease: 0 if the patient doesn't have any heart diseases, 1 if the patient has a heart disease
- ever_married: "No" or "Yes"
- work_type: "children", "Govt_jov", "Never_worked", "Private" or "Self-employed"
- Residence_type: "Rural" or "Urban"
- avg_glucose_level: average glucose level in blood
- bmi: body mass index
- smoking_status: "formerly smoked", "never smoked", "smokes" or "Unknown"*
- stroke: 1 if the patient had a stroke or 0 if not *Note: "Unknown" in smoking_status means that the information is unavailable for this patient

## Importing numpy and pandas

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

## Reading datasets.

In [2]:
train_df = pd.read_csv('/kaggle/input/playground-series-s3e2/train.csv')
test_df = pd.read_csv('/kaggle/input/playground-series-s3e2/test.csv')
submission = pd.read_csv('/kaggle/input/playground-series-s3e2/sample_submission.csv')
addition_data = pd.read_csv('/kaggle/input/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv')
# This additional data is from the original dataset

In [3]:
addition_data.bmi = addition_data.bmi.fillna(np.mean(addition_data.bmi))
# Filling na values with the mean of the column

In [4]:
train_df['generated'] = 1
test_df['generated'] = 1
addition_data['generated'] = 0
train_df = pd.concat([train_df, addition_data],axis=0, ignore_index=True)
# Data addition here just comes from the notebook https://www.kaggle.com/code/alexandershumilin/ps-s3-e2-ensemble-model-addition-data

### One hot encoding the dataset.

In [5]:
X_train = pd.get_dummies(train_df)
X_test = pd.get_dummies(test_df)

## Import modules

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


## Model selection
Candidates are, randomforest, svm regressor, catboost regressor

In [7]:
X = X_train.drop(["stroke"], axis=1)
y = X_train["stroke"]

# split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# create the models
rf = RandomForestRegressor()
svm = SVR()
cat = CatBoostRegressor()

# fit the models on the training data
rf.fit(X_train, y_train)
svm.fit(X_train, y_train)
cat.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_pred_svm = svm.predict(X_test)
y_pred_cat = cat.predict(X_test)

# calculate the mean squared error of each model on the test data
mse_rf = mean_squared_error(y_test, y_pred_rf)
mse_svm = mean_squared_error(y_test, y_pred_svm)
mse_cat = mean_squared_error(y_test, y_pred_cat)

# print the mean squared error of each model
print("Random Forest MSE:", mse_rf)
print("SVM MSE:", mse_svm)
print("CatBoost MSE:", mse_cat)

Learning rate set to 0.063655
0:	learn: 0.2016127	total: 60.3ms	remaining: 1m
1:	learn: 0.2003074	total: 63.5ms	remaining: 31.7s
2:	learn: 0.1991140	total: 66.7ms	remaining: 22.2s
3:	learn: 0.1981011	total: 69.8ms	remaining: 17.4s
4:	learn: 0.1970812	total: 73.1ms	remaining: 14.6s
5:	learn: 0.1962265	total: 76.4ms	remaining: 12.6s
6:	learn: 0.1954154	total: 79.6ms	remaining: 11.3s
7:	learn: 0.1946947	total: 82.9ms	remaining: 10.3s
8:	learn: 0.1939873	total: 86.1ms	remaining: 9.48s
9:	learn: 0.1934429	total: 89.8ms	remaining: 8.88s
10:	learn: 0.1928878	total: 92.9ms	remaining: 8.35s
11:	learn: 0.1924122	total: 96ms	remaining: 7.9s
12:	learn: 0.1919577	total: 99.1ms	remaining: 7.52s
13:	learn: 0.1914667	total: 102ms	remaining: 7.2s
14:	learn: 0.1910257	total: 106ms	remaining: 6.95s
15:	learn: 0.1905980	total: 109ms	remaining: 6.7s
16:	learn: 0.1902199	total: 112ms	remaining: 6.49s
17:	learn: 0.1898564	total: 115ms	remaining: 6.29s
18:	learn: 0.1895656	total: 118ms	remaining: 6.12s
19:	le

## Import of randsearch and metrics

In [8]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, accuracy_score

## Searching params for best model

In [9]:
param_grid = {
    'iterations': np.linspace(50, 200, 4, dtype=int),
    'depth': np.linspace(1, 8, 4, dtype=int),
    'learning_rate': np.linspace(0.01, 0.2, 4),
    'l2_leaf_reg':np.linspace(1,100,4,dtype=int)
}

cat = CatBoostRegressor()
random_search = RandomizedSearchCV(cat, param_grid, n_iter=20, cv=5, verbose=2, random_state=42)
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
0:	learn: 0.2034154	total: 7.25ms	remaining: 1.08s
1:	learn: 0.2032586	total: 13.9ms	remaining: 1.03s
2:	learn: 0.2030531	total: 16.3ms	remaining: 797ms
3:	learn: 0.2028789	total: 22.1ms	remaining: 806ms
4:	learn: 0.2026950	total: 28.6ms	remaining: 831ms
5:	learn: 0.2025252	total: 34.2ms	remaining: 820ms
6:	learn: 0.2023652	total: 39.8ms	remaining: 813ms
7:	learn: 0.2022076	total: 45.1ms	remaining: 801ms
8:	learn: 0.2020264	total: 50.7ms	remaining: 795ms
9:	learn: 0.2018587	total: 53ms	remaining: 741ms
10:	learn: 0.2017052	total: 58.5ms	remaining: 739ms
11:	learn: 0.2015392	total: 63.8ms	remaining: 734ms
12:	learn: 0.2013865	total: 69.1ms	remaining: 728ms
13:	learn: 0.2012510	total: 79ms	remaining: 767ms
14:	learn: 0.2010863	total: 84.6ms	remaining: 761ms
15:	learn: 0.2009383	total: 90.1ms	remaining: 754ms
16:	learn: 0.2007792	total: 95.5ms	remaining: 747ms
17:	learn: 0.2006394	total: 101ms	remaining: 743ms
18:	learn: 0.2004

RandomizedSearchCV(cv=5,
                   estimator=<catboost.core.CatBoostRegressor object at 0x7f0fc532a110>,
                   n_iter=20,
                   param_distributions={'depth': array([1, 3, 5, 8]),
                                        'iterations': array([ 50, 100, 150, 200]),
                                        'l2_leaf_reg': array([  1,  34,  67, 100]),
                                        'learning_rate': array([0.01      , 0.07333333, 0.13666667, 0.2       ])},
                   random_state=42, verbose=2)

In [10]:
# Print the best parameters
print("Best parameters found: ",random_search.best_params_)

# Print the best accuracy
print("Best mse found: ",random_search.best_score_)

Best parameters found:  {'learning_rate': 0.07333333333333333, 'l2_leaf_reg': 34, 'iterations': 200, 'depth': 3}
Best mse found:  0.12391844970524499


## Preparing data for final model training

In [11]:
X_train = pd.get_dummies(train_df)
X_test_final = pd.get_dummies(test_df)

y_test = X_train["stroke"]
X_train = X_train.drop(["stroke"], axis=1)

## Training a model with best params

In [12]:
cat_best = CatBoostRegressor(iterations=200, depth=3, learning_rate=0.073333, l2_leaf_reg=34)

# Fit the final model on the entire training set
cat_best.fit(X_train, y_test)

0:	learn: 0.2016771	total: 2.55ms	remaining: 507ms
1:	learn: 0.2003302	total: 4.95ms	remaining: 490ms
2:	learn: 0.1992638	total: 7.13ms	remaining: 468ms
3:	learn: 0.1983697	total: 9.26ms	remaining: 454ms
4:	learn: 0.1974090	total: 11.2ms	remaining: 437ms
5:	learn: 0.1965885	total: 13.2ms	remaining: 426ms
6:	learn: 0.1958889	total: 15.3ms	remaining: 421ms
7:	learn: 0.1953002	total: 17.3ms	remaining: 416ms
8:	learn: 0.1948122	total: 19.6ms	remaining: 415ms
9:	learn: 0.1943873	total: 21.6ms	remaining: 411ms
10:	learn: 0.1939993	total: 23.7ms	remaining: 407ms
11:	learn: 0.1936137	total: 25.7ms	remaining: 403ms
12:	learn: 0.1932387	total: 27.6ms	remaining: 398ms
13:	learn: 0.1928976	total: 29.6ms	remaining: 394ms
14:	learn: 0.1926014	total: 31.7ms	remaining: 392ms
15:	learn: 0.1923125	total: 33.9ms	remaining: 389ms
16:	learn: 0.1920707	total: 36ms	remaining: 388ms
17:	learn: 0.1918087	total: 37.9ms	remaining: 384ms
18:	learn: 0.1916098	total: 40.1ms	remaining: 382ms
19:	learn: 0.1914579	tot

In [13]:
final_predictions = cat_best.predict(X_test_final)

In [14]:
final_predictions[:20]

array([ 4.28975851e-02,  1.93563120e-01, -6.00319520e-04,  5.75044876e-02,
        7.79908652e-03,  1.84495425e-02,  4.50761641e-05,  5.57101195e-02,
        7.45647512e-04,  2.73255725e-02,  1.54749625e-02,  1.87974991e-01,
       -4.33907127e-04,  4.99160532e-03,  3.59022253e-02, -1.23387234e-04,
       -5.79154421e-04,  2.54283552e-03,  3.30537273e-02,  4.47103433e-02])

In [15]:
def norm_0to1(preds):
    return (preds - np.min(preds)) / (np.max(preds) - np.min(preds))

In [16]:
final_predictions = norm_0to1(final_predictions)

In [17]:
final_predictions[:20]

array([0.0982473 , 0.37817288, 0.01743137, 0.12538586, 0.03303685,
       0.05282462, 0.01863046, 0.12205206, 0.01993207, 0.06931564,
       0.04729807, 0.36779055, 0.01774055, 0.02782076, 0.08525043,
       0.01831747, 0.01747069, 0.02327112, 0.07995813, 0.10161528])

# Submission

In [18]:
sub = pd.DataFrame({'id': X_test_final['id'], 'stroke':final_predictions})

In [19]:
sub.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10204 entries, 0 to 10203
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      10204 non-null  int64  
 1   stroke  10204 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 159.6 KB


In [20]:
sub.to_csv('submission.csv', index=False)